<a href="https://www.kaggle.com/code/rajeshdaruru/1-financial-sentiment-fine-tune-notebook?scriptVersionId=351727330" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

**1. Imports and GPU check**

In [1]:
import os, json, collections
import pandas as pd, numpy as np, torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


**2. Load and prep the Financial PhraseBank dataset**

In [2]:
raw = load_dataset("FinanceMTEB/financial_phrasebank")
label_names = ["negative", "neutral", "positive"]  # verified label int <-> label_text mapping
print("Labels:", label_names)

train_df = raw["train"].to_pandas().rename(columns={"text": "sentence"})[["sentence", "label"]]
valid_df = raw["test"].to_pandas().rename(columns={"text": "sentence"})[["sentence", "label"]]
train_df = train_df.copy(); train_df["is_valid"] = False
valid_df = valid_df.copy(); valid_df["is_valid"] = True
full_df = pd.concat([train_df, valid_df]).reset_index(drop=True)

print(f"Train: {len(train_df)}  Valid: {len(valid_df)}")
full_df.head()

README.md:   0%|          | 0.00/465 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/104k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/80.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1264 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Labels: ['negative', 'neutral', 'positive']
Train: 1264  Valid: 1000


,sentence,label,is_valid
0,The Samsung Mobile Applications Store was laun...,1,False
1,"F-Secure , a developer of security solutions a...",1,False
2,The company serves customers in various indust...,1,False
3,The company reported net sales of 302 mln euro...,1,False
4,Microsoft last week also issued the first patc...,1,False


## Dataset attribution

This notebook uses the Financial PhraseBank dataset, created by Malo et al. Licensed under
CC BY-NC-SA 3.0 — non-commercial use only; for commercial use, contact the original authors.

> Malo, P., Sinha, A., Korhonen, P., Wallenius, J., & Takala, P. (2014). Good debt or bad debt:
> Detecting semantic orientations in economic texts. *Journal of the Association for Information
> Science and Technology*, 65(4), 782–796. https://doi.org/10.1002/asi.23062

**3. Train — Hugging Face `transformers`**

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

pretrained_model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name)

def tokenize(batch):
    return tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=128)

train_ds = Dataset.from_pandas(train_df[["sentence", "label"]]).map(tokenize, batched=True)
valid_ds = Dataset.from_pandas(valid_df[["sentence", "label"]]).map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name, num_labels=len(label_names)
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": (preds == labels).mean()}

args = TrainingArguments(
    output_dir="/kaggle/working/hf-run",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=valid_ds,
    compute_metrics=compute_metrics,
)
trainer.train()
print(trainer.evaluate())

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1264 [00:00<?, ? examples/s]

...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, 

Epoch,Training Loss,Validation Loss,Accuracy
1,1.273387,1.025295,0.777000
2,0.725370,0.623011,0.892000
3,0.411062,0.404205,0.940000
4,0.264346,0.354058,0.945000


{'eval_loss': 0.3540584146976471, 'eval_accuracy': 0.945, 'eval_runtime': 1.9428, 'eval_samples_per_second': 514.719, 'eval_steps_per_second': 8.236, 'epoch': 4.0}


**4. Save the model portable Hugging Face Format**

In [4]:
out_dir = "/kaggle/working/financial-sentiment-model"
os.makedirs(out_dir, exist_ok=True)
model.save_pretrained(out_dir)
tokenizer.save_pretrained(out_dir)
with open(f"{out_dir}/label_names.json", "w") as f:
    json.dump(label_names, f)
print("Saved to", out_dir)
print("Files will appear under this notebook's Output tab for download once the notebook finishes running.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /kaggle/working/financial-sentiment-model
Files will appear under this notebook's Output tab for download once the notebook finishes running.


**5. Predict using saved-and-reloaded model**

In [5]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

reloaded_model = AutoModelForSequenceClassification.from_pretrained("/kaggle/working/financial-sentiment-model").to(model.device).eval()
reloaded_tokenizer = AutoTokenizer.from_pretrained("/kaggle/working/financial-sentiment-model")

test_sentences = [
    "The company's quarterly earnings beat analyst expectations, sending shares higher.",
    "Revenue fell sharply amid weak demand and rising costs.",
    "The board approved a routine dividend payment in line with last quarter.",
    "Despite record profits, the CEO warned of a challenging outlook ahead.",  # mixed signal — good stress test
    "The firm announced layoffs while simultaneously reporting its best year on record.",  # ambiguous on purpose
]

for text in test_sentences:
    inputs = reloaded_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(reloaded_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = reloaded_model(**inputs).logits

    probs = torch.softmax(logits, dim=-1)[0]
    pred_label = label_names[probs.argmax().item()]
    confidence = probs.max().item()

    print(f"{pred_label:>10}  ({confidence:.1%} confident)  |  {text}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

  positive  (86.8% confident)  |  The company's quarterly earnings beat analyst expectations, sending shares higher.
  negative  (82.7% confident)  |  Revenue fell sharply amid weak demand and rising costs.
   neutral  (96.4% confident)  |  The board approved a routine dividend payment in line with last quarter.
  negative  (62.9% confident)  |  Despite record profits, the CEO warned of a challenging outlook ahead.
  positive  (43.1% confident)  |  The firm announced layoffs while simultaneously reporting its best year on record.
